# Week 3 Summary — Feature Engineering + Baseline

## Engineered features added
- tenure_bucket: grouped Tenure into New (0-1yr) / Established (1-3yr) / Loyal (3yr+)
- avg_monthly_spend: Total Charges / Tenure (with tenure=0 handled to avoid divide-by-zero)
- services_count: count of add-on services (0-6) per customer

## Class imbalance handling
- Used class_weight='balanced' in Logistic Regression instead of SMOTE, to avoid
  training on synthetic/fake data points for a baseline model.

## Baseline model: Logistic Regression
- Features scaled with StandardScaler (fit on train only, to avoid leakage into test set)
- F1 Score: 0.627
- ROC-AUC: 0.852
- Recall on churners: 0.80 (catches most churners, at the cost of some false alarms -
  precision 0.52) - deliberate trade-off given class_weight='balanced', appropriate
  since missing a churner is costlier than a wasted retention offer.
- Confusion matrix: 75 missed churners (false negatives) - the costliest error type,
  target for improvement in Week 4 with tree-based models.

This baseline is the benchmark Week 4's Random Forest/XGBoost models must beat.

In [19]:
import pandas as pd
df = pd.read_csv("../data/processed_customer_churn.csv")
print(df.shape)
df.columns.tolist()


(7043, 30)


['LoyaltyID',
 'Customer ID',
 'Senior Citizen',
 'Partner',
 'Dependents',
 'Tenure',
 'Phone Service',
 'Multiple Lines',
 'Internet Service',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Monthly Charges',
 'Total Charges',
 'Churn',
 'Age',
 'Married',
 'Number of Dependents',
 'Satisfaction Score',
 'CLTV',
 'Churn Category',
 'Churn Reason',
 'Zip Code',
 'Population']

# Categorize the columns in differnt headings to remeber every column

In [20]:

target_col = 'Churn'

drop_cols = ['LoyaltyID', 'Customer ID', 'Zip Code', 
             'Satisfaction Score', 'Churn Category', 'Churn Reason',
             'Dependents']  

categorical_cols = ['Senior Citizen', 'Partner', 'Phone Service', 'Multiple Lines',
                     'Internet Service', 'Online Security', 'Online Backup', 
                     'Device Protection', 'Tech Support', 'Streaming TV', 
                     'Streaming Movies', 'Contract', 'Paperless Billing', 
                     'Payment Method', 'Married']

numeric_cols = ['Tenure', 'Monthly Charges', 'Total Charges', 'Age', 
                'Number of Dependents', 'CLTV', 'Population']


all_accounted = set([target_col] + drop_cols + categorical_cols + numeric_cols)
all_actual = set(df.columns)
print("Missing from our lists:", all_actual - all_accounted)
print("Extra in our lists (typos?):", all_accounted - all_actual)

Missing from our lists: set()
Extra in our lists (typos?): set()


In [21]:
# check for any dupliaction  answers wrtiien just explained version
for col in categorical_cols:
    uniques = df[col].unique()
    if any('No internet' in str(v) or 'No phone' in str(v) for v in uniques):
        print(col, "->", uniques)

Multiple Lines -> <StringArray>
['No phone service', 'No', 'Yes']
Length: 3, dtype: str
Online Security -> <StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
Online Backup -> <StringArray>
['Yes', 'No', 'No internet service']
Length: 3, dtype: str
Device Protection -> <StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
Tech Support -> <StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
Streaming TV -> <StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
Streaming Movies -> <StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str


In [22]:
#  collapse 
pseudo_category_cols = ['Online Security', 'Online Backup', 'Device Protection',
                         'Tech Support', 'Streaming TV', 'Streaming Movies']

for col in pseudo_category_cols:
    df[col] = df[col].replace('No internet service', 'No')

df['Multiple Lines'] = df['Multiple Lines'].replace('No phone service', 'No')


for col in pseudo_category_cols + ['Multiple Lines']:
    print(col, "->", df[col].unique())

Online Security -> <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Online Backup -> <StringArray>
['Yes', 'No']
Length: 2, dtype: str
Device Protection -> <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Tech Support -> <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Streaming TV -> <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Streaming Movies -> <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Multiple Lines -> <StringArray>
['No', 'Yes']
Length: 2, dtype: str


# Identify the categorical coulumns to chenge to numerical valuesa s a model only work on numerical values

In [23]:

for col in categorical_cols:
    print(col, "->", df[col].nunique(), df[col].unique())

Senior Citizen -> 2 <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Partner -> 2 <StringArray>
['Yes', 'No']
Length: 2, dtype: str
Phone Service -> 2 <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Multiple Lines -> 2 <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Internet Service -> 3 <StringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str
Online Security -> 2 <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Online Backup -> 2 <StringArray>
['Yes', 'No']
Length: 2, dtype: str
Device Protection -> 2 <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Tech Support -> 2 <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Streaming TV -> 2 <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Streaming Movies -> 2 <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Contract -> 3 <StringArray>
['Month-to-month', 'One year', 'Two year']
Length: 3, dtype: str
Paperless Billing -> 2 <StringArray>
['Yes', 'No']
Length: 2, dtype: str
Payment Method -> 4 <StringArray>
[         'Electroni

# Transform Binary coulumns(YES/NO)

In [24]:

binary_cols = ['Senior Citizen', 'Partner', 'Phone Service', 'Multiple Lines',
               'Online Security', 'Online Backup', 'Device Protection',
               'Tech Support', 'Streaming TV', 'Streaming Movies',
               'Paperless Billing', 'Married']

for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})


df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})


df[binary_cols + ['Churn']].head()

,Senior Citizen,Partner,Phone Service,Multiple Lines,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Paperless Billing,Married,Churn
0,0,1,0,0,0,1,0,0,0,0,1,1,0
1,0,0,1,0,1,0,1,0,0,0,0,0,0
2,0,0,1,0,1,1,0,0,0,0,1,0,1
3,0,0,0,0,1,0,1,1,0,0,0,0,0
4,0,0,1,0,0,0,0,0,0,0,1,0,1


In [25]:

multi_cat_cols = ['Internet Service', 'Contract', 'Payment Method']

df = pd.get_dummies(df, columns=multi_cat_cols, drop_first=True)

print(df.shape)
df.columns.tolist()

(7043, 34)


['LoyaltyID',
 'Customer ID',
 'Senior Citizen',
 'Partner',
 'Dependents',
 'Tenure',
 'Phone Service',
 'Multiple Lines',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Paperless Billing',
 'Monthly Charges',
 'Total Charges',
 'Churn',
 'Age',
 'Married',
 'Number of Dependents',
 'Satisfaction Score',
 'CLTV',
 'Churn Category',
 'Churn Reason',
 'Zip Code',
 'Population',
 'Internet Service_Fiber optic',
 'Internet Service_No',
 'Contract_One year',
 'Contract_Two year',
 'Payment Method_Credit card (automatic)',
 'Payment Method_Electronic check',
 'Payment Method_Mailed check']

In [26]:
#drop the columns we decided not to use as features
df_model = df.drop(columns=drop_cols)

print(df_model.shape)
df_model.columns.tolist()

(7043, 27)


['Senior Citizen',
 'Partner',
 'Tenure',
 'Phone Service',
 'Multiple Lines',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Paperless Billing',
 'Monthly Charges',
 'Total Charges',
 'Churn',
 'Age',
 'Married',
 'Number of Dependents',
 'CLTV',
 'Population',
 'Internet Service_Fiber optic',
 'Internet Service_No',
 'Contract_One year',
 'Contract_Two year',
 'Payment Method_Credit card (automatic)',
 'Payment Method_Electronic check',
 'Payment Method_Mailed check']

In [27]:

X = df_model.drop(columns=['Churn'])
y = df_model['Churn']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\ny value counts:")
print(y.value_counts())

X shape: (7043, 26)
y shape: (7043,)

y value counts:
Churn
0    5174
1    1869
Name: count, dtype: int64


In [28]:

df_model.to_csv("../data/model_ready_churn.csv", index=False)
print("Saved:", df_model.shape)

Saved: (7043, 27)


# Tenure bucket

In [29]:

def bucket_tenure(t):
    if t <= 12:
        return 'New (0-1yr)'
    elif t <= 36:
        return 'Established (1-3yr)'
    else:
        return 'Loyal (3yr+)'

df_model['tenure_bucket'] = df['Tenure'].apply(bucket_tenure)
df_model = pd.get_dummies(df_model, columns=['tenure_bucket'], drop_first=True)
print(df_model.filter(like='tenure_bucket').sum())

tenure_bucket_Loyal (3yr+)    3001
tenure_bucket_New (0-1yr)     2186
dtype: int64


In [30]:
# avg monthly spend
df_model['avg_monthly_spend'] = (df['Total Charges'] / df['Tenure'].replace(0, 1)).round(2)
df_model['avg_monthly_spend'].describe()

count    7043.000000
mean       64.698184
std        30.270690
min         0.000000
25%        35.650000
50%        70.300000
75%        90.170000
max       121.400000
Name: avg_monthly_spend, dtype: float64

In [31]:
#  services_count
service_cols = ['Online Security', 'Online Backup', 'Device Protection',
                 'Tech Support', 'Streaming TV', 'Streaming Movies']
df_model['services_count'] = df_model[service_cols].sum(axis=1)
df_model['services_count'].value_counts().sort_index()

services_count
0    2219
1     966
2    1033
3    1118
4     852
5     571
6     284
Name: count, dtype: int64

# Train/test split and baseeline model 

In [32]:
from sklearn.model_selection import train_test_split
X = df_model.drop(columns = ['Churn'])
Y = df_model['Churn']
x_train,x_test,y_train,y_test = train_test_split(
    X,Y,test_size = 0.2,random_state  = 42 ,stratify=y)

print("Train shape:", x_train.shape)
print("Test shape:", x_test.shape)
print("Train churn rate:", y_train.mean().round(3))
print("Test churn rate:", y_test.mean().round(3))

Train shape: (5634, 30)
Test shape: (1409, 30)
Train churn rate: 0.265
Test churn rate: 0.265


# Logistic regresssion base line 


In [33]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter = 1000,class_weight = 'balanced',random_state = 42)
model.fit(x_train,y_train)
print("Model trained")

Model trained


/Users/RajatSaini1/telco-churn-prediction/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [39]:

from sklearn.preprocessing import StandardScaler

numeric_features = ['Tenure', 'Monthly Charges', 'Total Charges', 'Age', 
                     'Number of Dependents', 'CLTV', 'Population', 'avg_monthly_spend']

scaler = StandardScaler()
X_train_scaled = x_train.copy()
X_test_scaled = x_test.copy()

X_train_scaled[numeric_features] = scaler.fit_transform(x_train[numeric_features])
X_test_scaled[numeric_features] = scaler.transform(x_test[numeric_features])

print("Scaling done.")

Scaling done.


In [40]:

from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model.fit(X_train_scaled, y_train)
print("Model trained.")

Model trained.


In [41]:

from sklearn.metrics import f1_score, roc_auc_score, classification_report, confusion_matrix

y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"F1 Score: {f1:.3f}")
print(f"ROC-AUC: {roc_auc:.3f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

F1 Score: 0.627
ROC-AUC: 0.852

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.73      0.81      1035
           1       0.52      0.80      0.63       374

    accuracy                           0.75      1409
   macro avg       0.71      0.76      0.72      1409
weighted avg       0.80      0.75      0.76      1409


Confusion Matrix:
[[754 281]
 [ 75 299]]
